# Mapping figures

This notebook reproduces the mapping figures used in the manuscript.

It assumes all required files are located in one folder that is downloaded from the repository.
Place this notebook in that same folder, then run the notebook from top to bottom.

Outputs are written to an `outputs/` subfolder.

Figures produced:
1. Coal grades only
2. All sites
3. Literature sites only
4. Datashed.org sites only
5. Back-validation sites only

## 1. Libraries
Run this cell first. It imports all libraries used in this notebook.

In [42]:
from pathlib import Path
import warnings

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

try:
    from svgpath2mpl import parse_path
    import xml.dom.minidom as minidom
    SVG_SUPPORT = True
except Exception:
    SVG_SUPPORT = False

In [51]:
POINT_ALPHA = 0.8  # Adjust between 0.4–0.8 as desired


## 2. Folder layout and required files

This notebook uses a single folder that contains all required inputs.

Expected files in the same folder as this notebook:

- `Coal_Fields_in_Pennsylvania_High-Volatile_Bituminous.geojson`
- `Coal_Fields_in_Pennsylvania_Medium-Volatile_Bituminous.geojson`
- `Coal_Fields_in_Pennsylvania_Low-Volatile_Bituminous.geojson`
- `Coal_Fields_in_Pennsylvania_Semi-Anthracite.geojson`
- `Coal_Fields_in_Pennsylvania_Anthracite.geojson`
- `Pennsylvania_County_Boundaries.geojson`
- `All_Sites.xlsx` (must contain sheets: `Datashed`, `Backvalidation`, `Literature`)
- Optional: `backval_marker.svg` (custom marker for back-validation sites)

If your repository uses different filenames, edit the paths in the next cell.

In [52]:
BASE_DIR = Path.cwd()

OUTPUTS_DIR = BASE_DIR / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

COAL_FILES = {
    "High-Volatile Bituminous":  BASE_DIR / "Coal_Fields_in_Pennsylvania_High-Volatile_Bituminous.geojson",
    "Medium-Volatile Bituminous": BASE_DIR / "Coal_Fields_in_Pennsylvania_Medium-Volatile_Bituminous.geojson",
    "Low-Volatile Bituminous":   BASE_DIR / "Coal_Fields_in_Pennsylvania_Low-Volatile_Bituminous.geojson",
    "Semi-Anthracite":           BASE_DIR / "Coal_Fields_in_Pennsylvania_Semi-Anthracite.geojson",
    "Anthracite":                BASE_DIR / "Coal_Fields_in_Pennsylvania_Anthracite.geojson",
}

COUNTIES_FILE = BASE_DIR / "Pennsylvania_County_Boundaries.geojson"

ALL_SITES_XLSX = next((BASE_DIR / n for n in ["All_Sites.xlsx","All_Sites_TEST.xlsx","All_Sites_Test.xlsx"] if (BASE_DIR / n).exists()), BASE_DIR / "All_Sites.xlsx")  # sheets: Datashed, Backvalidation, Literature


BACKVAL_SVG = BASE_DIR / "mining-svgrepo-com.svg"  # optional

def require_file(p: Path):
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {p.resolve()}")

for p in list(COAL_FILES.values()) + [COUNTIES_FILE, ALL_SITES_XLSX]:
    require_file(p)

print("BASE_DIR:", BASE_DIR.resolve())
print("Outputs:", OUTPUTS_DIR.resolve())

BASE_DIR: /Users/rowanterra/Desktop/Test_1217/mapping
Outputs: /Users/rowanterra/Desktop/Test_1217/mapping/outputs


## 3. Load spatial layers and site tables

In [53]:
coal_gdfs = []
for rank, path in COAL_FILES.items():
    gdf = gpd.read_file(path)
    gdf["Rank"] = rank
    coal_gdfs.append(gdf)

coal = gpd.GeoDataFrame(pd.concat(coal_gdfs, ignore_index=True))


counties = gpd.read_file(COUNTIES_FILE)

TARGET_CRS = "EPSG:4326"
if coal.crs is None:
    coal = coal.set_crs(TARGET_CRS)
else:
    coal = coal.to_crs(TARGET_CRS)

if counties.crs is None:
    counties = counties.set_crs(TARGET_CRS)
else:
    counties = counties.to_crs(TARGET_CRS)


datashed_df = pd.read_excel(ALL_SITES_XLSX, sheet_name="Datashed")
backval_df  = pd.read_excel(ALL_SITES_XLSX, sheet_name="Backvalidation")
paper_df    = pd.read_excel(ALL_SITES_XLSX, sheet_name="Literature")

for _df in (datashed_df, backval_df, paper_df):
    _df["Lat"]  = pd.to_numeric(_df["Lat"], errors="coerce")
    _df["Long"] = pd.to_numeric(_df["Long"], errors="coerce")

# Define "unique site" by Lat/Long
COORD_DECIMALS = 6
def add_coord_key(df):
    df = df.copy()
    df["Lat_r"]  = df["Lat"].round(COORD_DECIMALS)
    df["Long_r"] = df["Long"].round(COORD_DECIMALS)
    df["coord_key"] = list(zip(df["Lat_r"], df["Long_r"]))
    return df

datashed_df = add_coord_key(datashed_df)
backval_df  = add_coord_key(backval_df)
paper_df    = add_coord_key(paper_df)

# Datashed.org: should not have repeated sites (but enforce uniqueness defensively)
datashed_df_unique = datashed_df.dropna(subset=["Lat_r","Long_r"]).drop_duplicates(subset=["coord_key"])

# Backvalidation: keep as-is for "sample count", but also make a unique-site version for mapping if needed
backval_df_unique  = backval_df.dropna(subset=["Lat_r","Long_r"]).drop_duplicates(subset=["coord_key"])

# Literature: keep all rows as "samples", but de-duplicate for mapping by Lat/Long
paper_df_unique    = paper_df.dropna(subset=["Lat_r","Long_r"]).drop_duplicates(subset=["coord_key"])

# GeoDataFrames used for plotting (use UNIQUE site tables)
datashed_gdf = gpd.GeoDataFrame(
    datashed_df_unique,
    geometry=gpd.points_from_xy(datashed_df_unique["Long"], datashed_df_unique["Lat"]),
    crs="EPSG:4326"
).to_crs(TARGET_CRS)

backval_gdf = gpd.GeoDataFrame(
    backval_df_unique,
    geometry=gpd.points_from_xy(backval_df_unique["Long"], backval_df_unique["Lat"]),
    crs="EPSG:4326"
).to_crs(TARGET_CRS)

paper_gdf = gpd.GeoDataFrame(
    paper_df_unique,
    geometry=gpd.points_from_xy(paper_df_unique["Long"], paper_df_unique["Lat"]),
    crs="EPSG:4326"
).to_crs(TARGET_CRS)

# Counts for legend / reporting
n_ds_sites = int(len(datashed_df_unique))
n_bv_sites = int(len(backval_df_unique))

n_lit_samples = int(paper_df.dropna(subset=["Lat_r","Long_r"]).shape[0])
n_lit_sites   = int(len(paper_df_unique))

print(f"Literature Samples n={n_lit_samples:,} (Unique sites n={n_lit_sites:,})")
print(f"Datashed.org Sites n={n_ds_sites:,}")
print(f"Back-validation Sites n={n_bv_sites:,}")


Literature Samples n=252 (Unique sites n=175)
Datashed.org Sites n=286
Back-validation Sites n=37


In [54]:


def overlap_report(df_a, name_a, df_b, name_b, cols=("Site", "County")):
    a = df_a.dropna(subset=["coord_key"]).copy()
    b = df_b.dropna(subset=["coord_key"]).copy()
    shared = set(a["coord_key"]).intersection(set(b["coord_key"]))
    if not shared:
        print(f"No shared coordinates between {name_a} and {name_b}.")
        return None

    aa = a[a["coord_key"].isin(shared)][["coord_key","Lat_r","Long_r"] + [c for c in cols if c in a.columns]].copy()
    bb = b[b["coord_key"].isin(shared)][["coord_key","Lat_r","Long_r"] + [c for c in cols if c in b.columns]].copy()
    aa["Sheet"] = name_a
    bb["Sheet"] = name_b
    out = pd.concat([aa, bb], ignore_index=True).sort_values(["Lat_r","Long_r","Sheet"])
    return out

pairs = [
    ("Literature", paper_df, "Datashed", datashed_df),
    ("Literature", paper_df, "Backvalidation", backval_df),
    ("Datashed", datashed_df, "Backvalidation", backval_df),
]

reports = []
for a_name, a_df, b_name, b_df in pairs:
    rep = overlap_report(a_df, a_name, b_df, b_name)
    if rep is not None:
        print(f"Shared coordinates between {a_name} and {b_name}: {rep['coord_key'].nunique():,}")
        reports.append(rep)

if reports:
    overlap_df = pd.concat(reports, ignore_index=True)
    display(overlap_df)


No shared coordinates between Literature and Datashed.
No shared coordinates between Literature and Backvalidation.
Shared coordinates between Datashed and Backvalidation: 31


,coord_key,Lat_r,Long_r,Site,County,Sheet
0,"(40.249304, -79.765766)",40.249304,-79.765766,Lowber,Westmoreland,Backvalidation
1,"(40.249304, -79.765766)",40.249304,-79.765766,Lowber,Westmoreland,Datashed
2,"(40.366808, -78.645556)",40.366808,-78.645556,Puritian,Cambria,Backvalidation
3,"(40.366808, -78.645556)",40.366808,-78.645556,Puritan Discharge,Cambria,Datashed
4,"(40.382222, -80.089444)",40.382222,-80.089444,Scrubgrass,Allegheny,Backvalidation
...,...,...,...,...,...,...
57,"(41.204722, -79.165833)",41.204722,-79.165833,Filson 2 and 3,Jefferson,Datashed
58,"(41.2295, -79.9431)",41.229500,-79.943100,Sterrett,Venango,Backvalidation
59,"(41.2295, -79.9431)",41.229500,-79.943100,Sterrett,Venango,Datashed
60,"(41.364444, -79.239444)",41.364444,-79.239444,Hefren,Clarion,Backvalidation


## 4. Styling

Site colors are fixed as:

- Literature sites: black
- Datashed.org sites: green
- Back-validation sites: orange

Coal polygons are filled by rank and drawn without bold outlines.
County boundaries are drawn in black.

In [65]:
# Coal fill colors
RANK_COLORS = {
    "High-Volatile Bituminous":  "#c7dbf0",
    "Medium-Volatile Bituminous": "#8ab6e6",
    "Low-Volatile Bituminous":   "#4f85c5",
    "Semi-Anthracite":           "#d7c3ef",
    "Anthracite":                "#6d4fa8",
}

# Fixed site colors
SITE_COLORS = {
    "literature": "black",
    "datashed": "green",
    "back_validation": "orange",
}

# Marker settings
SIZE = 20
BACKVAL_SIZE = int(SIZE * 4)  # back-validation marker a bit larger
ALPHA = 0.95

# Polygon outline controls 
COAL_EDGE_COLOR = "none"
COAL_EDGE_WIDTH = 0.0

# Counties outline controls
COUNTY_EDGE_COLOR = "black"
COUNTY_EDGE_WIDTH = 0.8

# Back validation marker
USE_SVG_BACKVAL = True

def svg_to_marker(svg_path: Path):
    """Parse an SVG and return a Matplotlib Path suitable for use as a marker.

    Notes:
    - Many SVG icons are defined in a screen coordinate system where Y increases downward.
      Matplotlib uses Y increasing upward, so we flip the Y axis.
    - SVG paths can have arbitrary scale; we normalize the path to a ~unit box so marker sizing is stable.
    """
    doc = minidom.parse(str(svg_path))
    path_strings = [p.getAttribute("d") for p in doc.getElementsByTagName("path")]
    doc.unlink()

    if len(path_strings) == 0:
        raise ValueError("No <path> elements found in the SVG. Use an SVG with <path d='...'> elements.")

    mpl_paths = [parse_path(d) for d in path_strings]
    from matplotlib.path import Path as MplPath

    if len(mpl_paths) == 1:
        path = mpl_paths[0]
    else:
        path = MplPath.make_compound_path(*mpl_paths)

    verts = path.vertices.copy()
    verts -= verts.mean(axis=0)

    verts[:, 1] *= -1

    x_span = verts[:, 0].max() - verts[:, 0].min()
    y_span = verts[:, 1].max() - verts[:, 1].min()
    span = max(x_span, y_span)
    if span == 0:
        raise ValueError("SVG path span is zero; cannot normalize marker.")
    verts /= span

    path.vertices = verts
    return path

BACKVAL_MARKER = ">"  # fallback if SVG not used
BACKVAL_SVG_MARKER = None

if USE_SVG_BACKVAL and SVG_SUPPORT and BACKVAL_SVG.exists():
    try:
        BACKVAL_SVG_MARKER = svg_to_marker(BACKVAL_SVG)
        print("Using SVG marker for back-validation sites:", BACKVAL_SVG.name)
    except Exception as e:
        warnings.warn(f"Could not use SVG marker ({e}); falling back to built-in marker.")
else:
    if USE_SVG_BACKVAL and not BACKVAL_SVG.exists():
        print("SVG marker not found (optional):", BACKVAL_SVG.name, " -> using built-in marker instead.")

Using SVG marker for back-validation sites: mining-svgrepo-com.svg


## 5. Plotting helpers

Legend layout is forced into three columns:

- Column 1: Bituminous grades (high, medium, low)
- Column 2: Anthracite grades (semi-anthracite, anthracite)
- Column 3: Site categories (back-validation, literature, datashed)

A blank spacer is included so the site categories stay in their own column.

In [66]:
def build_legend_groups(n_backval: int, n_lit_samples: int, n_lit_sites: int, n_ds_sites: int):
    """Return three legend handle lists: bituminous, anthracite, data."""
    # Coal patches
    hv = Patch(facecolor=RANK_COLORS["High-Volatile Bituminous"], edgecolor="black", label="High-Volatile Bituminous")
    mv = Patch(facecolor=RANK_COLORS["Medium-Volatile Bituminous"], edgecolor="black", label="Medium-Volatile Bituminous")
    lv = Patch(facecolor=RANK_COLORS["Low-Volatile Bituminous"], edgecolor="black", label="Low-Volatile Bituminous")
    sa = Patch(facecolor=RANK_COLORS["Semi-Anthracite"], edgecolor="black", label="Semi-Anthracite")
    an = Patch(facecolor=RANK_COLORS["Anthracite"], edgecolor="black", label="Anthracite")

    # Site handles
    back_marker = BACKVAL_SVG_MARKER if BACKVAL_SVG_MARKER is not None else BACKVAL_MARKER

    h_lit = Line2D([0], [0], marker="o", linestyle="",
                   markerfacecolor=SITE_COLORS["literature"], markeredgecolor=SITE_COLORS["literature"],
                   markersize=10, label=f"Literature Samples, n={n_lit_samples} ({n_lit_sites} unique sites)")
    h_ds = Line2D([0], [0], marker="o", linestyle="",
                  markerfacecolor=SITE_COLORS["datashed"], markeredgecolor=SITE_COLORS["datashed"],
                  markersize=10, label=f"Datashed.org Sites, n={n_ds_sites}")
    h_bv = Line2D([0], [0], marker=back_marker, linestyle="",
                  markerfacecolor=SITE_COLORS["back_validation"], markeredgecolor=SITE_COLORS["back_validation"],
                  markersize=20, label=f"Back-validation Sites, n={n_backval}")

    bituminous = [lv, mv, hv]          # low, medium, high
    anthracite = [an, sa]              # anthracite, semi-anthracite
    data = [h_lit, h_ds, h_bv]         # literature, datashed, back-validation
    return bituminous, anthracite, data


def add_three_column_legend(fig, ax, n_backval: int, n_lit: int, n_ds: int):
    """Add three separate legends aligned as columns below the plot."""
    bituminous, anthracite, data = build_legend_groups(n_backval, n_lit, n_ds)

    # Common styling
    legend_kwargs = dict(frameon=False, handletextpad=0.8, borderaxespad=0.0, fontsize=14)

    # Place three legends under the axis.
    # Use figure coordinates so spacing is stable across exports.
    leg1 = fig.legend(handles=bituminous, ncol=1, loc="lower left",
                      bbox_to_anchor=(0.10, 0.02), **legend_kwargs)
    leg2 = fig.legend(handles=anthracite, ncol=1, loc="lower left",
                      bbox_to_anchor=(0.44, 0.02), **legend_kwargs)
    leg3 = fig.legend(handles=data, ncol=1, loc="lower left",
                      bbox_to_anchor=(0.72, 0.02), **legend_kwargs)

    # Ensure legends stay on top
    for leg in (leg1, leg2, leg3):
        leg.set_zorder(10)

## 6. Generate figures
Run this cell to write all figures to the outputs folder.

In [67]:
# This cell defines all functions used to build and export the figures.

def plot_base(ax):
    # Counties boundary
    counties.boundary.plot(ax=ax, linewidth=COUNTY_EDGE_WIDTH, color=COUNTY_EDGE_COLOR)

    # Coal polygons by rank (fills, no bold outlines)
    for rank, color in RANK_COLORS.items():
        subset = coal[coal["Rank"] == rank]
        if not subset.empty:
            subset.plot(
                ax=ax,
                color=color,
                edgecolor=COAL_EDGE_COLOR,
                linewidth=COAL_EDGE_WIDTH,
                alpha=0.8
            )


def plot_sites(ax, mode: str):
    """mode in {'all','literature','datashed','back_validation'}"""
    if mode in ("all", "literature"):
        paper_gdf.plot(ax=ax, color=SITE_COLORS["literature"], markersize=SIZE, alpha=ALPHA, marker="o")

    if mode in ("all", "datashed"):
        datashed_gdf.plot(ax=ax, color=SITE_COLORS["datashed"], markersize=SIZE, alpha=ALPHA, marker="o")

    if mode in ("all", "back_validation"):
        marker = BACKVAL_SVG_MARKER if BACKVAL_SVG_MARKER is not None else BACKVAL_MARKER
        # Use ax.scatter() so the custom path marker renders as a crisp vector shape.
        xs = backval_gdf.geometry.x.values
        ys = backval_gdf.geometry.y.values
        ax.scatter(xs, ys, marker=marker, s=BACKVAL_SIZE * 4,
           color=SITE_COLORS["back_validation"], alpha=POINT_ALPHA, zorder=5,
           edgecolors="black", linewidths=0.5)


def add_legend(ax, n_backval, n_lit_samples, n_lit_sites, n_ds_sites):
    """Three ax.legend() calls placed inside the axes as separate columns.

    Legends are anchored in axes-fraction coordinates so they always stay
    within the map extent regardless of bbox_inches="tight".
    """
    # Column 1, Bituminous grades
    col1 = [
        Patch(facecolor=RANK_COLORS["Low-Volatile Bituminous"],    edgecolor="black", label="Low-Volatile Bituminous"),
        Patch(facecolor=RANK_COLORS["Medium-Volatile Bituminous"],  edgecolor="black", label="Medium-Volatile Bituminous"),
        Patch(facecolor=RANK_COLORS["High-Volatile Bituminous"],    edgecolor="black", label="High-Volatile Bituminous"),
    ]

    # Column 2, Anthracite grades
    col2 = [
        Patch(facecolor=RANK_COLORS["Anthracite"],      edgecolor="black", label="Anthracite"),
        Patch(facecolor=RANK_COLORS["Semi-Anthracite"], edgecolor="black", label="Semi-Anthracite"),
    ]

    # Column 3, Site categories
    back_marker = BACKVAL_SVG_MARKER if BACKVAL_SVG_MARKER is not None else BACKVAL_MARKER
    col3 = [
        Line2D([0],[0], marker="o", linestyle="",
               markerfacecolor=SITE_COLORS["literature"], markeredgecolor=SITE_COLORS["literature"],
               markersize=9, label=f"Literature Samples, n={n_lit_samples} ({n_lit_sites} unique sites)"),
        Line2D([0],[0], marker="o", linestyle="",
               markerfacecolor=SITE_COLORS["datashed"], markeredgecolor=SITE_COLORS["datashed"],
               markersize=9, label=f"Datashed.org Sites, n={n_ds_sites}"),
        Line2D([0],[0], marker=back_marker, linestyle="", markerfacecolor=SITE_COLORS["back_validation"],
               markeredgecolor="black", markeredgewidth=0.5,
               markersize=15, label=f"Back-validation Sites, n={n_backval}"),
    ]

    common = dict(
        frameon=False,
        facecolor="white",
        edgecolor="none",
        fontsize=10,
        handletextpad=0.5,
        labelspacing=0.5,
        handlelength=1.8,
        borderpad=0.4,
        borderaxespad=0.0,
        labelcolor="black",
        loc="lower left",
    )


    leg1 = ax.legend(handles=col1, bbox_to_anchor=(0.05, -0.10), **common)
    ax.add_artist(leg1)

    leg2 = ax.legend(handles=col2, bbox_to_anchor=(0.35, -0.10), **common)
    ax.add_artist(leg2)

    leg3 = ax.legend(handles=col3, bbox_to_anchor=(0.54, -0.10), **common)


def make_figure(mode, filename, add_legend_flag=True):
    fig, ax = plt.subplots(figsize=(12, 7), dpi=300)

    fig.patch.set_alpha(0)
    ax.patch.set_alpha(0)

    plot_base(ax)

    if mode != "coal_only":
        plot_sites(ax, mode)

    ax.set_axis_off()

    if add_legend_flag:
        add_legend(ax, n_backval, n_lit_samples, n_lit_sites, n_ds_sites)

    out = OUTPUTS_DIR / filename
    fig.savefig(out, dpi=300, bbox_inches="tight", transparent=True)
    plt.close(fig)
    return out


# Site Counts

# Literature
n_lit_samples = len(paper_df)                 # total rows
n_lit_sites   = len(paper_df_unique)          # unique lat/long sites

# Datashed.org
n_ds_sites    = len(datashed_df_unique)

# Backvalidation
n_backval     = len(backval_df_unique)

print("Literature Samples:", n_lit_samples)
print("Literature Unique Sites:", n_lit_sites)
print("Datashed.org Sites:", n_ds_sites)
print("Backvalidation Sites:", n_backval)


Literature Samples: 273
Literature Unique Sites: 175
Datashed.org Sites: 286
Backvalidation Sites: 37


In [68]:
written = []
written.append(make_figure("coal_only", "01_coal_grades_only.png", add_legend_flag=True))
written.append(make_figure("all", "02_all_sites.png", add_legend_flag=True))
written.append(make_figure("literature", "03_literature_only.png", add_legend_flag=True))
written.append(make_figure("datashed", "04_datashed_only.png", add_legend_flag=True))
written.append(make_figure("back_validation", "05_back_validation_only.png", add_legend_flag=True))

print("Wrote:")
for p in written:
    print(" -", p.resolve())

Wrote:
 - /Users/rowanterra/Desktop/Test_1217/mapping/outputs/01_coal_grades_only.png
 - /Users/rowanterra/Desktop/Test_1217/mapping/outputs/02_all_sites.png
 - /Users/rowanterra/Desktop/Test_1217/mapping/outputs/03_literature_only.png
 - /Users/rowanterra/Desktop/Test_1217/mapping/outputs/04_datashed_only.png
 - /Users/rowanterra/Desktop/Test_1217/mapping/outputs/05_back_validation_only.png
